### MAST Request for coronagraphic observation

In [12]:
import sys
import os
import time
import re
import json
 
import requests
from urllib.parse import quote as urlencode

from astropy.table import Table
import numpy as np

import pprint
pp = pprint.PrettyPrinter(indent=4)

In [3]:
def mast_query(request):
    """Perform a MAST query.
 
        Parameters
        ----------
        request (dictionary): The MAST request json object
 
        Returns head,content where head is the response HTTP headers, and content is the returned data"""
 
    # Base API url
    request_url='https://mast.stsci.edu/api/v0/invoke'
 
    # Grab Python Version
    version = ".".join(map(str, sys.version_info[:3]))
 
    # Create Http Header Variables
    headers = {"Content-type": "application/x-www-form-urlencoded",
               "Accept": "text/plain",
               "User-agent":"python-requests/"+version}
 
    # Encoding the request as a json string
    req_string = json.dumps(request)
    req_string = urlencode(req_string)
 
    # Perform the HTTP request
    resp = requests.post(request_url, data="request="+req_string, headers=headers)
 
    # Pull out the headers and response content
    head = resp.headers
    content = resp.content.decode('utf-8')
 
    return head, content


def download_request(payload, filename, download_type="file"):
    request_url='https://mast.stsci.edu/api/v0.1/Download/' + download_type
    resp = requests.post(request_url, data=payload)
 
    with open(filename,'wb') as FLE:
        FLE.write(resp.content)
 
    return filename

def set_filters(parameters):
    return [{"paramName":p, "values":v} for p,v in parameters.items()]

def set_min_max(min, max):
    return [{'min': min, 'max': max}]

def mast_json2csv(json):
    csv_str =  ",".join([x['name'] for x in json['fields']])
    csv_str += "\n"
    csv_str += ",".join([x['type'] for x in json['fields']])
    csv_str += "\n"
 
    col_names = [x['name'] for x in json['fields']]
    for row in json['data']:
        csv_str += ",".join([str(row.get(col,"nul")) for col in col_names]) + "\n"
 
    return csv_str



 
def mast_json2table(json_obj):
 
    data_table = Table()
 
    for col,atype in [(x['name'],x['type']) for x in json_obj['fields']]:
        if atype=="string":
            atype="str"
        if atype=="boolean":
            atype="bool"
        data_table[col] = np.array([x.get(col,None) for x in json_obj['data']],dtype=atype)
 
    return data_table


def resolve_name(object_of_interest):
 
    request = {'service':'Mast.Name.Lookup',
               'params':{'input':object_of_interest,
                         'format':'json'},
    }
 
    headers, out_string = mast_query(request)
 
    out_data = json.loads(out_string)
 
    return out_data



In [10]:
resolved_object = resolve_name('HR 8799')


obj_ra = resolved_object['resolvedCoordinate'][0]['ra']
obj_dec = resolved_object['resolvedCoordinate'][0]['decl']



mast_request = {'service':'Mast.Caom.Cone',
                'params':{'ra':obj_ra,
                          'dec':obj_dec,
                          'radius':0.2},
                'format':'json',
                'pagesize':2000,
                'page':1,
                'removenullcolumns':True,
                'removecache':True}

headers, mast_data_str = mast_query(mast_request)

mast_data = json.loads(mast_data_str)

print(mast_data.keys())
print("Query status:",mast_data['status'])



dict_keys(['status', 'msg', 'data', 'fields', 'paging'])
Query status: COMPLETE


In [13]:
pp.pprint(mast_data['fields'][:5])

[   {'name': 'intentType', 'type': 'string'},
    {'name': 'obs_collection', 'type': 'string'},
    {'name': 'provenance_name', 'type': 'string'},
    {'name': 'instrument_name', 'type': 'string'},
    {'name': 'project', 'type': 'string'}]


In [16]:
pp.pprint(mast_data['data'][10])

{   'calib_level': 2,
    'dataRights': 'PUBLIC',
    'dataURL': 'https://irsa.ipac.caltech.edu/data/SPITZER/SHA/archive/proc/IRSX002500/r3565568/ch2/bcd/SPITZER_S2_3565568_0022_0000_11_bcd.fits',
    'dataproduct_type': 'spectrum',
    'distance': 0,
    'em_max': 38000,
    'em_min': 14000,
    'filters': 'IRS-LL',
    'instrument_name': 'IRS',
    'intentType': 'science',
    'jpegURL': None,
    'mtFlag': False,
    'obs_collection': 'SPITZER_SHA',
    'obs_id': '000000366800',
    'obs_title': 'Spectroscopy of protostellar, protoplanetary and debris '
                 'disks',
    'obsid': 1677691,
    'project': None,
    'proposal_id': 'IRS_DISKS_2',
    'proposal_pi': 'Houck, James R.',
    'proposal_type': 'Spitzer',
    'provenance_name': 'SSC Pipeline',
    's_dec': 21.13425,
    's_ra': 346.869625,
    's_region': 'POLYGON 346.885459 21.14172 346.886653 21.138965 346.787276 '
                '21.101462 346.786081 21.104217 346.885459 21.14172',
    'sequence_number': 356556

In [17]:


mast_data_table = Table()

for col,atype in [(x['name'],x['type']) for x in mast_data['fields']]:
    if atype=="string":
        atype="str"
    if atype=="boolean":
        atype="bool"
    mast_data_table[col] = np.array([x.get(col,None) for x in mast_data['data']],dtype=atype)
    
print(mast_data_table)



intentType obs_collection provenance_name ...   obsid        distance     
---------- -------------- --------------- ... --------- ------------------
   science           TESS            SPOC ... 100152130                0.0
   science           TESS            SPOC ...  97937494                0.0
   science           TESS            SPOC ...  97937517                0.0
   science    SPITZER_SHA    SSC Pipeline ...   1677691                0.0
   science    SPITZER_SHA    SSC Pipeline ...   1677691                0.0
   science    SPITZER_SHA    SSC Pipeline ...   1677691                0.0
   science    SPITZER_SHA    SSC Pipeline ...   1677691                0.0
       ...            ...             ... ...       ...                ...
   science    SPITZER_SHA    SSC Pipeline ...   1721993  35.12975019908253
   science    SPITZER_SHA    SSC Pipeline ...   1721993 35.388122649877744
   science    SPITZER_SHA    SSC Pipeline ...   1721993 35.388122649877744
   science    SPITZER_SHA

In [26]:
mashup_request = {"service":"Mast.Caom.Filtered",
                  "format":"json",
                  "params":{
                      "columns":"COUNT_BIG(*)",
                      "filters":[
                          {"paramName":"filters",
                           "values":["NUV","FUV"],
                           "separator":";"
                          },
                          {"paramName":"t_max",
                           "values":[{"min":52264.4586,"max":53452.8914}], #MJD
                          },
                          {"paramName":"obsid",
                           "values":[],
                           "freeText":"%200%"}
                      ]}}
    
headers, out_string = mast_query(mashup_request)
count = json.loads(out_string)

pp.pprint(count)

{   'data': [{'Column1': 255}],
    'fields': [{'name': 'Column1', 'type': 'string'}],
    'msg': '',
    'paging': {   'page': 1,
                  'pageSize': 1,
                  'pagesFiltered': 1,
                  'rows': 1,
                  'rowsFiltered': 1,
                  'rowsTotal': 1},
    'status': 'COMPLETE'}


In [27]:


mashup_request = {"service":"Mast.Caom.Filtered",
                 "format":"json",
                 "params":{
                     "columns":"*",
                     "filters":[
                         {"paramName":"filters",
                          "values":["NUV","FUV"],
                          "separator":";"
                         },
                         {"paramName":"t_max",
                          "values":[{"min":52264.4586,"max":54452.8914}], #MJD
                         },
                         {"paramName":"obsid",
                          "values":[],
                          "freeText":"%200%"}
                     ]}}
    
headers, out_string = mast_query(mashup_request)
filtered_data = json.loads(out_string)

print(filtered_data.keys())
print("Query status:", filtered_data['status'])



dict_keys(['status', 'msg', 'data', 'fields', 'paging'])
Query status: COMPLETE


In [28]:
filts = set_filters({"dataproduct_type": ["cube"]})

mashup_request = {
        "service":"Mast.Caom.Filtered.Position",
        "format":"json",
        "params":{
            "columns":"COUNT_BIG(*)",
            "filters":filts,
            "position":"210.8023, 54.349, 0.24"
        }}

headers, out_string = mast_query(mashup_request)
count = json.loads(out_string)

pp.pprint(count)


{   'data': [{'Column1': 858}],
    'fields': [{'name': 'Column1', 'type': 'string'}],
    'msg': '',
    'paging': {   'page': 1,
                  'pageSize': 1,
                  'pagesFiltered': 1,
                  'rows': 1,
                  'rowsFiltered': 1,
                  'rowsTotal': 1},
    'status': 'COMPLETE'}


In [29]:
# Picking the first Hubble Space Telescope observation
interesting_observation = mast_data_table[mast_data_table["obs_collection"] == "HST"][0]
print("Observation:",
      [interesting_observation[x] for x in ['dataproduct_type', 'obs_collection', 'instrument_name']])


IndexError: index 0 out of range for table with length 0

In [25]:
interesting_observation = mast_data_table[mast_data_table["obs_collection"] == "JWST"][0]
print("Observation:",
      [interesting_observation[x] for x in ['dataproduct_type', 'obs_collection', 'instrument_name']])

IndexError: index 0 out of range for table with length 0